In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = "v_commerce"
silver_schema_name = "silver"

In [0]:
tb_avaliacoes_bronze = spark.table("v_commerce.bronze.tb_avaliacoes")
tb_catalogo_produtos_bronze = spark.table("v_commerce.bronze.tb_catalogo_produtos")
tb_clickstream_bronze = spark.table("v_commerce.bronze.tb_clickstream")
tb_cliente_bronze = spark.table("v_commerce.bronze.tb_clientes")
tb_pedidos_bronze = spark.table("v_commerce.bronze.tb_pedidos")
tb_suporte_tickets_bronze = spark.table("v_commerce.bronze.tb_suporte_tickets")

In [0]:
window_spec = Window.partitionBy("id_evento").orderBy(F.col("timestamp_ingestion").desc())

column_dispositivo_treated = F.trim(F.lower(F.col("dispositivo")))
column_evento_treated = F.trim(F.lower(F.col("tipo_evento")))
column_canal_treated = F.regexp_replace(F.trim(F.lower(F.col("canal"))), r"\s+", "_")

tb_clickstream_silver = (
    tb_clickstream_bronze
    .select(
        F.col("id_evento"),
        F.col("id_sessao"),
        F.coalesce(F.col("id_cliente"), F.lit("usuario_anonimo")).alias("id_cliente"),
        F.col("id_dispositivo"),
        F.coalesce(F.col("id_produto"), F.lit("n_a")).alias("id_produto"),

        F.when(column_evento_treated.isin("pageview", "page-view"), F.lit("page_view"))
         .when(column_evento_treated.isin("busca", "srch"), F.lit("search"))
         .when(column_evento_treated.isin("adicionar", "add-to-cart", "addtocart"), F.lit("add_to_cart"))
         .when(column_evento_treated.isin("prod-view", "product-view", "productview", "prod_view"), F.lit("product_view"))
         .when(column_evento_treated.isin("abandon-cart", "abandoncart", "abandono", ), F.lit("abandon_cart"))
         .when(column_evento_treated.isin("login", "singin", "signin", "log-in", "sing-in"), F.lit("log_in"))
         .when(column_evento_treated.isin("compra", "buy"), F.lit("purchase"))
         .when(column_evento_treated.isin("pagamento"), F.lit("payment"))
         .when(column_evento_treated.isin("pv"), F.lit("no_specified"))
         .otherwise(column_evento_treated).alias("tipo_evento"),

        F.when(column_canal_treated.isin("aplicativo"), F.lit("app"))
         .otherwise(column_canal_treated).alias("canal"),

        F.when(column_dispositivo_treated.isin("computador"), F.lit("desktop"))
         .when(column_dispositivo_treated.isin("mob", "celular"), F.lit("mobile"))
         .when(column_dispositivo_treated.isin("tab"), F.lit("tablet"))
         .otherwise(column_dispositivo_treated).alias("dispositivo"),

        F.col("origem_sessao"),
        F.col("data_evento"),
        F.coalesce(F.col("tempo_pagina_seg"), F.lit(0)).alias("tempo_pagina_seg"),
        F.col("timestamp_ingestion"),
        F.current_timestamp().alias("timestamp_ingestion_silver")
    )
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

tb_clickstream_silver.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{silver_schema_name}.tb_clickstream")

In [0]:
window_spec_tickets = Window.partitionBy("id_ticket").orderBy(F.col("timestamp_ingestion").desc())

column_tipo_problema_treated = F.trim(F.lower(F.col("tipo_problema")))
column_agente_suporte_treated = F.initcap(F.trim(F.lower(F.col("agente_suporte"))))

tb_suporte_tickets_silver = (
    tb_suporte_tickets_bronze
    .select(
        F.col("ticket_id").alias('id_ticket'),
        F.col("id_cliente"),
        F.col("id_pedido"),
        
        # 1. Tratamento do tipo_problema (agrupando pt, en e erros de digitação)
        F.when(column_tipo_problema_treated.isin("pro", "produto", "p3oduto", "product", "prod"), F.lit("product"))
         .when(column_tipo_problema_treated.isin("pagamento", "p4gamento", "pag", "payment", "pay"), F.lit("payment"))
         .when(column_tipo_problema_treated.isin("entrega", "3ntrega", "entr"), F.lit("delivery"))
         .when(column_tipo_problema_treated.isin("ref", "reembolso", "r3embolso", "reemb", "refund"), F.lit("refund"))
         .when(column_tipo_problema_treated.isin("del", "delay", ), F.lit("delay"))
         .otherwise(column_tipo_problema_treated).alias("tipo_problema"),
        
        F.col("data_abertura"),
        
        # Nulos mantidos para representar que os tickets não foram resolvidos ainda
        F.col("data_resolucao"),
        F.col("tempo_resolucao_horas"),
        
        column_agente_suporte_treated.alias("agente_suporte"),
        
        # Os nulos serão substituídos pelo valor -1 para não alterar a nota ao mesmo tempo que não fica como nulo
        F.coalesce(F.col("nota_avaliacao"), F.lit(-1)).alias("nota_avaliacao"),
        
        F.col("timestamp_ingestion"),
        F.current_timestamp().alias("timestamp_ingestion_silver")
    )
    .withColumn("row_number", F.row_number().over(window_spec_tickets))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "timestamp_ingestion")
)

tb_suporte_tickets_silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{silver_schema_name}.fato_suporte_tickets")